<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week1/lesson4/Lesson4_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 4 — Exercises**
## Matrix Transformations: from Neural Layers to Behaviour

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 1 — Linear Algebra I\
**Lesson:** Lesson 4

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | The matrix–vector product, three ways | 10 |
| 2 | Matrix–matrix products and batches | 15 |
| 3 | Building transformations | 15 |
| 4 | Composition, rank and inverses | 15 |
| 5 | Affine maps and a neural-network layer | 15 |
| 6 | Neural and behavioural coordinates: a simulation | 15 |
| 7 | Real data: how many neurons does a decoder need? | 15 |
| **Total** | | **100** |
| Bonus | Rotations preserve dot products | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)   # exercise notebook uses its own fixed seed
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def rotation(deg):
    # 2x2 matrix that rotates by `deg` degrees anticlockwise (from the teaching notebook)
    t = np.radians(deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

print("Setup complete.")

---
## Exercise 1 — The Matrix–Vector Product, Three Ways (10 points)

Four presynaptic neurons project to three postsynaptic neurons. Row $i$ of `W_ex` holds the weights onto postsynaptic neuron $i$; column $j$ belongs to presynaptic neuron $j$ (the Lesson 3 convention).

In [ ]:
W_ex = np.array([[ 0.2,  0.5, -0.4,  0.0],
                 [ 0.7, -0.1,  0.3,  0.2],
                 [-0.3,  0.4,  0.6, -0.5]])   # shape (3, 4): (postsynaptic, presynaptic), no units
r_in = np.array([10.0, 4.0, 6.0, 8.0])       # shape (4,): presynaptic rates, spikes/s

**1.1 (3 points, written response, no code).** What are the shapes of `W_ex` and of $\mathbf{z} = \mathbf{W}\mathbf{r}$? What does the entry $w_{23} = 0.3$ mean (which neuron to which, and what a positive sign says)? Why would `W_ex @ np.ones(3)` fail?

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (4 points): Compute z = W_ex r_in three ways and check they agree.
#   z_rows -- row view: a loop of dot products, one per row of W_ex
#   z_cols -- column view: the sum over j of r_in[j] * (column j of W_ex)
#   z_at   -- W_ex @ r_in

# YOUR CODE HERE
z_rows = None
z_cols = None
z_at = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert z_rows is not None and z_cols is not None and z_at is not None, "z_rows, z_cols and z_at are still None."
assert np.shape(z_rows) == np.shape(z_cols) == np.shape(z_at) == (3,), "Each result should have one value per postsynaptic neuron: shape (3,)."
assert np.allclose(z_rows, z_at) and np.allclose(z_cols, z_at), "The three views should give the same vector."
assert np.allclose(z_at, W_ex @ r_in), "z_at should be W_ex @ r_in."
print("Self-check passed.")

**1.3 (3 points, written response).** Use the column view, without recomputing the whole product, to say how $\mathbf{z}$ changes if presynaptic neuron 3 doubles its rate from 6 to 12 spikes/s. Which postsynaptic neuron's input falls?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Matrix–Matrix Products and Batches (15 points)

In [ ]:
A2 = RNG.normal(size=(2, 3))
B2 = RNG.normal(size=(3, 4))
C2 = RNG.normal(size=(4, 2))
X_batch = RNG.poisson(lam=[10.0, 4.0, 6.0, 8.0], size=(20, 4)).astype(float)   # shape (20, 4): (trial, presynaptic), spikes/s
P2 = np.array([[1.0, 2.0], [0.0, 1.0]])
Q2 = np.array([[0.0, 1.0], [1.0, 0.0]])

In [ ]:
# 2.1 (4 points): Without running the products, fill in the shape of each result as a tuple,
# or None if the product is not defined (inner sizes do not match).
#   A2 is (2, 3), B2 is (3, 4), C2 is (4, 2)

shapes = {
    'A2 @ B2': ___,
    'B2 @ C2': ___,
    'A2 @ B2 @ C2': ___,
    'C2 @ A2': ___,
    'B2 @ A2': ___,
}

In [ ]:
# SELF-CHECK 2.1 -- compares your predictions with what NumPy does, without printing the answers.
assert isinstance(shapes, dict) and len(shapes) == 5, "Fill in all five entries of shapes."
assert all(v is None or isinstance(v, tuple) for v in shapes.values()), "Replace every ___ with a tuple such as (2, 4), or None."
def _actual(expr):
    try:
        return eval(expr).shape
    except ValueError:
        return None
wrong = [k for k, v in shapes.items() if (tuple(v) if v is not None else None) != _actual(k)]
assert not wrong, f"Check these predictions again: {wrong}"
print("Self-check passed.")

In [ ]:
# 2.2 (4 points): Send all 20 trials of X_batch through W_ex (from Exercise 1) with ONE matrix product.
#   Z_batch -- shape (20, 3): row t is W_ex applied to trial t
# Then check it against a loop over trials (Z_loop) with np.allclose.

# YOUR CODE HERE
Z_batch = None
Z_loop = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert Z_batch is not None and Z_loop is not None, "Z_batch and Z_loop are still None."
assert np.shape(Z_batch) == (20, 3), "Z_batch should have one row per trial and one column per postsynaptic neuron: (20, 3)."
assert np.allclose(Z_batch, Z_loop), "The batch product and the loop should agree."
assert np.allclose(Z_batch[0], W_ex @ X_batch[0]), "Row 0 of Z_batch should be W_ex applied to trial 0."
print("Self-check passed.")

In [ ]:
# 2.3 (4 points): For the 2x2 matrices P2 and Q2:
#   PQ, QP     -- the two products P2 @ Q2 and Q2 @ P2
#   commute    -- True if they are equal (np.allclose), False otherwise
#   associative -- True if (P2 @ Q2) @ A2[:, :2].T equals P2 @ (Q2 @ A2[:, :2].T)

# YOUR CODE HERE
PQ, QP = None, None
commute = None
associative = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.3
assert PQ is not None and QP is not None and commute is not None and associative is not None, "Fill in PQ, QP, commute and associative."
assert np.allclose(PQ, P2 @ Q2) and np.allclose(QP, Q2 @ P2), "PQ and QP should be the two products, in that order."
assert isinstance(commute, (bool, np.bool_)) and isinstance(associative, (bool, np.bool_)), "commute and associative should be True or False."
print("Self-check passed.")

**2.4 (3 points, written response).** $\mathbf{Q}_2$ swaps the two coordinates of a vector. Use that to explain in words why $\mathbf{P}_2\mathbf{Q}_2$ and $\mathbf{Q}_2\mathbf{P}_2$ differ (what does each do to the rows or columns of $\mathbf{P}_2$?). Then explain why the batch product in 2.2 is `X_batch @ W_ex.T` and not `W_ex @ X_batch`.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Building Transformations (15 points)

In [ ]:
# 3.1 (4 points): Write three functions that return 2x2 matrices.
#   shear_matrix(k)       -- horizontal shear: (x, y) -> (x + k y, y)
#   scale_matrix(sx, sy)  -- stretch x by sx and y by sy
#   projection_matrix(u)  -- projection onto the line through the vector u (u need not be unit length):
#                            first make it a unit vector u_hat, then return np.outer(u_hat, u_hat)

def shear_matrix(k):
    # YOUR CODE HERE
    raise NotImplementedError

def scale_matrix(sx, sy):
    # YOUR CODE HERE
    raise NotImplementedError

def projection_matrix(u):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1 -- tested on vectors, so the matrices themselves are not printed.
assert np.allclose(shear_matrix(2.0) @ np.array([1.0, 3.0]), [7.0, 3.0]), "shear_matrix(k) should send (x, y) to (x + k y, y)."
assert np.allclose(scale_matrix(2.0, 0.5) @ np.array([3.0, 4.0]), [6.0, 2.0]), "scale_matrix(sx, sy) should send (x, y) to (sx x, sy y)."
P_chk = projection_matrix([3.0, 4.0])
assert np.allclose(P_chk @ np.array([3.0, 4.0]), [3.0, 4.0]), "A vector already on the line should not move."
assert np.allclose(P_chk @ np.array([-4.0, 3.0]), [0.0, 0.0]), "A vector at right angles to the line should project to zero."
print("Self-check passed.")

In [ ]:
# 3.2 (4 points): Build the matrix M32 that sends e1 = (1, 0) to (2, 1) and e2 = (0, 1) to (-1, 1).
# (Hint: the columns of a matrix are where the basis vectors go.)
# Then compute:
#   square_image -- M32 applied to the corners of the unit square, given below as the COLUMNS of `corners`
#   area_factor  -- |det(M32)|, the area of the image of the unit square

corners = np.array([[0.0, 1.0, 1.0, 0.0],
                    [0.0, 0.0, 1.0, 1.0]])    # shape (2, 4): one corner per column

# YOUR CODE HERE
M32 = None
square_image = None
area_factor = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert M32 is not None and square_image is not None and area_factor is not None, "M32, square_image and area_factor are still None."
assert np.shape(M32) == (2, 2), "M32 should be 2x2."
assert np.allclose(M32 @ [1, 0], [2, 1]) and np.allclose(M32 @ [0, 1], [-1, 1]), "M32 should send e1 to (2, 1) and e2 to (-1, 1)."
assert np.shape(square_image) == (2, 4), "square_image should keep one corner per column: shape (2, 4)."
assert np.isclose(area_factor, abs(np.linalg.det(M32))), "area_factor should be |det(M32)|."
print("Self-check passed.")

In [ ]:
# 3.3 (4 points): For each transformation below, record two facts in the dict `props`:
#   'keeps_length' -- True if the length of v_test is unchanged (np.isclose of the norms)
#   'area'         -- the area factor |det|
# props should look like {'rotate 40': {'keeps_length': ..., 'area': ...}, ...}

v_test = np.array([0.6, 0.8])
transforms = {
    'rotate 40': rotation(40),
    'shear 0.5': shear_matrix(0.5),
    'scale (2, 0.5)': scale_matrix(2.0, 0.5),
    'project onto 45 deg': projection_matrix([1.0, 1.0]),
}

# YOUR CODE HERE
props = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.3
assert globals().get('props') is not None, "props is not filled in yet (3.3 also needs your functions from 3.1)."
assert isinstance(props, dict) and set(props) == set(transforms), "props needs one entry per transformation."
for name, M in transforms.items():
    assert set(props[name]) == {'keeps_length', 'area'}, f"{name}: record 'keeps_length' and 'area'."
    assert np.isclose(props[name]['area'], abs(np.linalg.det(M))), f"{name}: 'area' should be |det|."
print("Self-check passed.")

**3.4 (3 points, written response).** Which of the four transformations in 3.3 lose information, in the sense that two different inputs can give the same output? Explain using the area factor. Then explain in words why a projection matrix satisfies $\mathbf{P}\mathbf{P} = \mathbf{P}$ (check it numerically if you like).

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Composition, Rank and Inverses (15 points)

In [ ]:
# 4.1 (4 points): Let S4 = scale_matrix(2, 0.5) and R4 = rotation(90).
#   M_scale_then_rotate -- the single matrix that first scales, then rotates
#   M_rotate_then_scale -- the single matrix that first rotates, then scales
# Apply both to x4 = (1, 1) and store the outputs in y_sr and y_rs.

S4, R4 = scale_matrix(2.0, 0.5), rotation(90)
x4 = np.array([1.0, 1.0])

# YOUR CODE HERE
M_scale_then_rotate = None
M_rotate_then_scale = None
y_sr, y_rs = None, None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1 -- checks the order by applying the stages one at a time.
assert globals().get('M_scale_then_rotate') is not None, "Build both composite matrices (4.1 also needs scale_matrix from 3.1)."
assert M_scale_then_rotate is not None and M_rotate_then_scale is not None, "Build both composite matrices."
assert np.allclose(M_scale_then_rotate @ x4, R4 @ (S4 @ x4)), "M_scale_then_rotate should equal applying S4 first, then R4."
assert np.allclose(M_rotate_then_scale @ x4, S4 @ (R4 @ x4)), "M_rotate_then_scale should equal applying R4 first, then S4."
assert np.allclose(y_sr, M_scale_then_rotate @ x4) and np.allclose(y_rs, M_rotate_then_scale @ x4), "y_sr and y_rs should be the two outputs."
print("Self-check passed.")

In [ ]:
# 4.2 (4 points): Ranks.
#   ranks -- a dict with the rank (np.linalg.matrix_rank) of each matrix in `rank_cases`
#   chain_rank -- the rank of W_out4 @ W_in4, where 30 neurons project through 3 relay neurons onto 30 neurons

rank_cases = {
    'tall (3x2)': np.array([[1.0, 2.0], [0.0, 1.0], [1.0, 0.0]]),
    'repeated rows (3x3)': np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 0.0, 1.0]]),
    'projection onto 45 deg': projection_matrix([1.0, 1.0]),
    'rotate 40': rotation(40),
}
W_in4 = RNG.normal(size=(3, 30))
W_out4 = RNG.normal(size=(30, 3))

# YOUR CODE HERE
ranks = None
chain_rank = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert globals().get('ranks') is not None, "ranks is not filled in yet (4.2 also needs projection_matrix from 3.1)."
assert ranks is not None and chain_rank is not None, "ranks and chain_rank are still None."
assert set(ranks) == set(rank_cases), "ranks needs one entry per matrix."
assert all(ranks[k] == np.linalg.matrix_rank(M) for k, M in rank_cases.items()), "Use np.linalg.matrix_rank for each matrix."
assert chain_rank == np.linalg.matrix_rank(W_out4 @ W_in4), "chain_rank should be the rank of W_out4 @ W_in4."
print("Self-check passed.")

In [ ]:
# 4.3 (4 points): Write inverse_2x2(M) using the formula  (1 / (ad - bc)) [[d, -b], [-c, a]].
# If |ad - bc| < 1e-12, raise a ValueError("matrix is singular") instead.
# Then use it to check that the inverse of (R4 S4) equals inverse(S4) @ inverse(R4):
#   reverse_order_ok -- True or False

def inverse_2x2(M):
    # YOUR CODE HERE
    raise NotImplementedError

reverse_order_ok = None

In [ ]:
# SELF-CHECK 4.3
M_chk = np.array([[3.0, 1.0], [2.0, 4.0]])
assert np.allclose(inverse_2x2(M_chk) @ M_chk, np.eye(2)), "inverse_2x2(M) @ M should be the identity."
try:
    inverse_2x2(projection_matrix([1.0, 2.0]))
    raise AssertionError("inverse_2x2 should raise a ValueError for a singular matrix.")
except ValueError:
    pass
assert reverse_order_ok is not None, "Compute reverse_order_ok."
print("Self-check passed.")

**4.4 (3 points, written response).** In 4.2, explain the rank of the `repeated rows` matrix and the value of `chain_rank`. A readout matrix of shape $2\times30$ maps 30 neurons to 2 behavioural variables: can any matrix recover the 30 rates from its output? Why or why not?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Affine Maps and a Neural-Network Layer (15 points)

In [ ]:
relu = lambda z: np.maximum(0.0, z)
W1_5 = RNG.normal(size=(4, 3))     # layer 1: 3 inputs -> 4 units
b1_5 = RNG.normal(size=4)
W2_5 = RNG.normal(size=(2, 4))     # layer 2: 4 units -> 2 outputs
X5 = RNG.normal(size=(10, 3))      # 10 samples, one per row

In [ ]:
# 5.1 (4 points): Write layer(X, W, b, f) for a BATCH of samples X (one sample per row).
# It returns f(X W^T + b), shape (n_samples, n_units). If f is None, return the affine part X W^T + b.
# Compute A1 = layer(X5, W1_5, b1_5, relu).

def layer(X, W, b, f=None):
    # YOUR CODE HERE
    raise NotImplementedError

A1 = None   # replace

In [ ]:
# SELF-CHECK 5.1
assert A1 is not None, "A1 is still None."
assert np.shape(A1) == (10, 4), "A1 should have one row per sample and one column per unit: (10, 4)."
assert np.allclose(layer(X5[:1], W1_5, b1_5)[0], W1_5 @ X5[0] + b1_5), "Without f, layer should return X W^T + b."
assert np.all(A1 >= 0), "After ReLU every activation should be >= 0."
print("Self-check passed.")

In [ ]:
# 5.2 (4 points): Two layers WITHOUT biases.
#   collapse_linear -- True if W2_5 (W1_5 x) equals (W2_5 W1_5) x for every sample (use X5, one sample per row)
#   collapse_relu   -- True if W2_5 relu(W1_5 x) equals (W2_5 W1_5) x for every sample
#   max_gap         -- the largest absolute difference between W2_5 relu(W1_5 x) and (W2_5 W1_5) x over all samples

# YOUR CODE HERE
collapse_linear = None
collapse_relu = None
max_gap = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert collapse_linear is not None and collapse_relu is not None and max_gap is not None, "Fill in all three results."
assert isinstance(collapse_linear, (bool, np.bool_)) and isinstance(collapse_relu, (bool, np.bool_)), "The two collapse results should be True or False."
assert max_gap >= 0, "max_gap is an absolute difference, so it cannot be negative."
print("Self-check passed.")

In [ ]:
# 5.3 (4 points): The augmented-matrix trick. For W1_5 (4x3) and b1_5 (4,), build W_aug5 of shape (5, 4):
#   [[W1_5, b1_5 as a column],
#    [0 0 0, 1]]
# Check that W_aug5 @ (x, 1) reproduces (W1_5 x + b1_5, 1) for the first sample of X5:
#   aug_ok -- True or False

# YOUR CODE HERE
W_aug5 = None
aug_ok = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.3
assert W_aug5 is not None and aug_ok is not None, "W_aug5 and aug_ok are still None."
assert np.shape(W_aug5) == (5, 4), "W_aug5 should have shape (5, 4)."
assert np.allclose(W_aug5[-1], [0, 0, 0, 1]), "The last row of W_aug5 should be (0, 0, 0, 1)."
assert np.allclose(W_aug5 @ np.append(X5[3], 1.0), np.append(W1_5 @ X5[3] + b1_5, 1.0)), "W_aug5 @ (x, 1) should give (W x + b, 1)."
print("Self-check passed.")

**5.4 (3 points, written response).** Using your results from 5.2, explain why stacking ten linear layers gives a model no more powerful than one matrix, and what the ReLU changes. Then say why a model neuron's pre-activation $z$ may be negative while its activation should not be.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Neural and Behavioural Coordinates: a Simulation (15 points)

As in Part 8 of the teaching notebook, each of $N$ neurons is cosine-tuned to the movement direction $\mathbf{d} = (\cos\theta, \sin\theta)$: $r_i = b_i + g_i\cos(\theta - \phi_i)$, which is $\mathbf{r} = \mathbf{b} + \mathbf{E}\mathbf{d}$ with row $i$ of $\mathbf{E}$ equal to $g_i(\cos\phi_i,\ \sin\phi_i)$.

You will compare two simulated populations of 40 neurons:
- `phi_even`: preferred directions spread evenly round the circle;
- `phi_clustered`: preferred directions bunched within ±30° of the left–right axis.

In [ ]:
N6 = 40
phi_even = np.linspace(0, 2 * np.pi, N6, endpoint=False)
phi_clustered = np.radians(np.concatenate([RNG.uniform(-30, 30, N6 // 2), RNG.uniform(150, 210, N6 // 2)]))
g6 = RNG.uniform(5, 15, N6)                         # depths of modulation, spikes/s
b6 = RNG.uniform(10, 30, N6)                        # baselines, spikes/s
theta8 = np.radians(np.arange(0, 360, 45))          # the eight test directions
D8 = np.vstack([np.cos(theta8), np.sin(theta8)])    # shape (2, 8): one direction per column

In [ ]:
# 6.1 (4 points): Write encoding_matrix(phi, g) returning E, shape (N, 2), with row i = g_i (cos phi_i, sin phi_i).
# Then compute the noiseless rates for the eight directions in the clustered population:
#   r8 -- shape (40, 8): column k is b6 + E d_k, with d_k = column k of D8 (do it with ONE matrix product plus broadcasting)

def encoding_matrix(phi, g):
    # YOUR CODE HERE
    raise NotImplementedError

r8 = None   # replace

In [ ]:
# SELF-CHECK 6.1
assert r8 is not None, "r8 is still None."
E_chk = encoding_matrix(np.array([0.0, np.pi / 2]), np.array([2.0, 3.0]))
assert np.allclose(E_chk, [[2.0, 0.0], [0.0, 3.0]]), "encoding_matrix: row i should be g_i (cos phi_i, sin phi_i)."
assert np.shape(r8) == (40, 8), "r8 should have one row per neuron and one column per direction: (40, 8)."
assert np.allclose(r8[:, 2], b6 + encoding_matrix(phi_clustered, g6) @ D8[:, 2]), "Column k of r8 should be b6 + E d_k."
print("Self-check passed.")

In [ ]:
# 6.2 (4 points): Write popvec_decoder(phi, g) returning the 2 x N population-vector decoder,
#   column i = (2 / N) (cos phi_i, sin phi_i) / g_i
# Then compute the 2x2 round-trip matrices:
#   DE_even      -- popvec_decoder @ encoding_matrix for the evenly spread population
#   DE_clustered -- the same for the clustered population

def popvec_decoder(phi, g):
    # YOUR CODE HERE
    raise NotImplementedError

DE_even = None
DE_clustered = None

In [ ]:
# SELF-CHECK 6.2
assert DE_even is not None and DE_clustered is not None, "DE_even and DE_clustered are still None."
D_chk = popvec_decoder(np.array([0.0, np.pi / 2]), np.array([2.0, 4.0]))
assert np.shape(D_chk) == (2, 2) and np.allclose(D_chk, [[0.5, 0.0], [0.0, 0.25]]), "popvec_decoder: column i should be (2/N)(cos phi_i, sin phi_i)/g_i."
assert np.shape(DE_even) == (2, 2) and np.shape(DE_clustered) == (2, 2), "Both round-trip matrices are 2x2."
print("Self-check passed.")

In [ ]:
# 6.3 (4 points): Decode the eight directions for the CLUSTERED population from (r8 - b6) with
#   (a) the plain decoder D, and (b) the corrected decoder D_c = inv(D E) D.
# Store the wrapped angular errors in degrees (decoded minus true, in [-180, 180)) as
#   err_plain, err_corrected -- each shape (8,)

def wrap_deg(a_rad):
    return (np.degrees(a_rad) + 180) % 360 - 180

# YOUR CODE HERE
err_plain = None
err_corrected = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.3
assert err_plain is not None and err_corrected is not None, "err_plain and err_corrected are still None."
assert np.shape(err_plain) == (8,) and np.shape(err_corrected) == (8,), "One error per direction: shape (8,)."
assert np.all(np.abs(err_plain) <= 180) and np.all(np.abs(err_corrected) <= 180), "Wrap the errors into [-180, 180)."
print("Self-check passed.")

**6.4 (3 points, written response).** Describe `DE_even` and `DE_clustered`. Which directions does the plain decoder get wrong for the clustered population, and in which direction are they pulled? Why does the corrected decoder fix this completely here, when in the real data of Part 9 it helped only a little?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: How Many Neurons Does a Decoder Need? (15 points)

This exercise uses the motor-cortex recording from Lesson 2 and Part 9 of the teaching notebook: 171 neurons in a monkey's primary motor cortex during centre-out reaches to eight targets ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); shared through the [DREAM database on CRCNS](https://crcns.org/data-sets/movements/dream) and packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)).

Brain–computer interfaces record from tens to hundreds of neurons. How does decoding accuracy depend on how many you have? You will build the encoding and decoding matrices for random **subsets** of the well-tuned neurons and measure the error on the held-out half of the session.

The cell below downloads the file (1.3 MB) the first time it runs, fits the encoding matrix exactly as in Part 9, and selects the well-tuned neurons. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
m1 = loadmat(M1_FILE)
rates = m1['spikes'].astype(float) / 0.05                     # shape (171, 15536): (neuron, 50 ms bin), spikes/s
hand_vel = m1['handVel'][:2]
moving = np.linalg.norm(hand_vel, axis=0) > 0.1
direction_index = np.round((np.degrees(np.arctan2(hand_vel[1], hand_vel[0])) % 360) / 45).astype(int) % 8
first_half = np.arange(rates.shape[1]) < rates.shape[1] // 2

def tuning_table(use):
    return np.stack([rates[:, use & moving & (direction_index == k)].mean(axis=1) for k in range(8)], axis=1)

tuning_fit, tuning_test = tuning_table(first_half), tuning_table(~first_half)   # each (171, 8), spikes/s
base_all = tuning_fit.mean(axis=1)                                              # (171,): baselines, spikes/s
E_all = (tuning_fit - base_all[:, None]) @ D8.T / 4                             # (171, 2): encoding matrix
model_all = E_all @ D8
norms = np.linalg.norm(tuning_fit - base_all[:, None], axis=1) * np.linalg.norm(model_all, axis=1)
fit_sim = np.where(norms > 0, ((tuning_fit - base_all[:, None]) * model_all).sum(axis=1) / np.where(norms > 0, norms, 1), 0.0)
tuned_idx = np.flatnonzero(fit_sim > 0.8)                                       # indices of the well-tuned neurons
print(f"E_all: {E_all.shape};  well-tuned neurons: {tuned_idx.size}")

In [ ]:
# 7.1 (4 points): Write decoders(idx) for an array of neuron indices idx. Using the rows E_all[idx]:
#   g   -- the depth of each neuron: the length of its row of E_all
#   U   -- each row divided by its length (unit preferred directions)
#   D   -- the 2 x n population-vector decoder, column i = (2 / n) U_i / g_i   (n = len(idx))
#   D_c -- the corrected decoder inv(D E) D
# Return (D, D_c).

def decoders(idx):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
D_t, D_t_c = decoders(tuned_idx[:10])
assert np.shape(D_t) == (2, 10) and np.shape(D_t_c) == (2, 10), "For 10 neurons both decoders should be 2 x 10."
assert np.allclose(D_t_c @ E_all[tuned_idx[:10]], np.eye(2)), "The corrected decoder should satisfy D_c E = I."
print("Self-check passed.")

In [ ]:
# 7.2 (4 points): Write held_out_error(Dmat, idx) that decodes the eight TEST tuning curves of the neurons idx,
#   out = Dmat @ (tuning_test[idx] - base_all[idx][:, None])        -- shape (2, 8)
# and returns the mean absolute angular error in degrees over the eight directions
# (compare the angle of each column of out with theta8, wrapping errors into [-180, 180)).

def held_out_error(Dmat, idx):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2 -- uses its own decoder, so it does not depend on 7.1.
E_t = E_all[tuned_idx]
g_t = np.linalg.norm(E_t, axis=1)
D_ref = (2 / len(tuned_idx)) * (E_t / g_t[:, None]**2).T
e_chk = held_out_error(D_ref, tuned_idx)
assert np.ndim(e_chk) == 0, "held_out_error should return a single number."
assert 0 <= e_chk <= 180, "A mean absolute angular error lies between 0 and 180 degrees."
print("Self-check passed.")

In [ ]:
# 7.3 (4 points): For each subset size n in sizes, draw 30 random subsets of tuned_idx with
#   rng7.choice(tuned_idx, n, replace=False)
# (use the generator rng7 below, in the order: for each n, for each of the 30 repeats), and record the mean,
# over the 30 subsets, of the held-out error of the plain and the corrected decoder:
#   mean_err_plain, mean_err_corr -- each shape (len(sizes),)
# Then plot both against n (log-scaled x-axis), with axis labels, a title and a legend.

sizes = [3, 5, 10, 20, 40, 82]
rng7 = np.random.default_rng(7)

# YOUR CODE HERE
mean_err_plain = None
mean_err_corr = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert mean_err_plain is not None and mean_err_corr is not None, "mean_err_plain and mean_err_corr are still None."
assert np.shape(mean_err_plain) == (len(sizes),) and np.shape(mean_err_corr) == (len(sizes),), "One mean error per subset size."
assert np.all((np.asarray(mean_err_plain) >= 0) & (np.asarray(mean_err_plain) <= 180)), "Errors are angles between 0 and 180 degrees."
print("Self-check passed.")

**7.4 (3 points, written response — interpretation and limitation).** How does the error change as neurons are added, and why would you expect that? When does the correction $(\mathbf{D}\mathbf{E})^{-1}$ help, and why can it *hurt* with very few neurons? (Think about Part 6 and what $\mathbf{D}\mathbf{E}$ looks like for 3 neurons.) Give one limitation of this analysis.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Rotations Preserve Dot Products (extra credit, not part of the 100 points)

A rotation moves the plane rigidly, so it should keep every length and every angle, and therefore every dot product.

1. Verify numerically that $\mathbf{R}^{\mathsf T}\mathbf{R} = \mathbf{I}$ for `rotation(37)`.
2. Using $\mathbf{u}\cdot\mathbf{v} = \mathbf{u}^{\mathsf T}\mathbf{v}$ (Lesson 3) and the transpose rule $(\mathbf{R}\mathbf{u})^{\mathsf T} = \mathbf{u}^{\mathsf T}\mathbf{R}^{\mathsf T}$, show in writing that $(\mathbf{R}\mathbf{u})\cdot(\mathbf{R}\mathbf{v}) = \mathbf{u}\cdot\mathbf{v}$.
3. Check it on 1000 random pairs of vectors, and find one non-rotation matrix from this exercise notebook for which it fails.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` or `___` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.